# medfarm-llama v2: QLoRA fine-tune on the whole KB

Trains Llama 3.1 8B Instruct on `dataset_v2.jsonl` (built by `build_dataset.py`), checks it on
held-out questions and exports `medfarm-llama-Q4_K_M.gguf` for Ollama.

**Runtime → Change runtime type → T4 GPU.** About 40 minutes training + 15 minutes export.

Steps: 1) upload `dataset_v2.jsonl`, 2) load the model, 3) train, 4) check, 5) save to Google Drive.

In [ ]:
%%capture
!pip install unsloth

## 1. Dataset
Upload `backend/models/chat_bot/dataset_v2.jsonl`.

In [ ]:
import json
from google.colab import files

uploaded = files.upload()  # choose dataset_v2.jsonl
rows = [json.loads(line) for line in open("dataset_v2.jsonl", encoding="utf-8")]
train_rows = [r for r in rows if r["split"] == "train"]
test_rows = [r for r in rows if r["split"] == "test"]
print(len(train_rows), "train /", len(test_rows), "test")

## 2. Model
Same base and LoRA settings as the first model (MODEL_CARD.md).

In [ ]:
from unsloth import FastLanguageModel

MAX_LEN = 4096  # Sinhala needs several tokens per character; matches the server's num_ctx
model, tokenizer = FastLanguageModel.from_pretrained(
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit", max_seq_length=MAX_LEN, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=3407)

## 3. Train
The text is exactly what Ollama sends with the Modelfile template (user turn = prompt, no system
message), and the loss is on the answer only.

In [ ]:
from datasets import Dataset
from trl import SFTConfig, SFTTrainer
from unsloth import is_bf16_supported
from unsloth.chat_templates import train_on_responses_only

USER = "<|start_header_id|>user<|end_header_id|>\n\n"
ASSISTANT = "<|start_header_id|>assistant<|end_header_id|>\n\n"


def chat(prompt, answer=""):
    text = f"<|begin_of_text|>{USER}{prompt}<|eot_id|>{ASSISTANT}"
    return text + (f"{answer}<|eot_id|>" if answer else "")


lengths = [len(tokenizer(chat(r["prompt"], r["answer"]), add_special_tokens=False).input_ids)
           for r in train_rows]
print("tokens: max", max(lengths), "mean", sum(lengths) // len(lengths))
assert max(lengths) <= MAX_LEN, "raise MAX_LEN"

dataset = Dataset.from_list([{"text": chat(r["prompt"], r["answer"])} for r in train_rows])
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text", per_device_train_batch_size=2, gradient_accumulation_steps=4,
        num_train_epochs=1, learning_rate=2e-4, warmup_steps=10, lr_scheduler_type="linear",
        optim="adamw_8bit", weight_decay=0.01, logging_steps=5, seed=3407,
        fp16=not is_bf16_supported(), bf16=is_bf16_supported(),
        output_dir="outputs", report_to="none"))
trainer = train_on_responses_only(trainer, instruction_part=USER, response_part=ASSISTANT)
stats = trainer.train()
print(stats)

## 4. Check on held-out questions
A sample of the test split (Sinhala generation is slow on a T4). Same checks as `eval_ollama.py`:
right behaviour (answer / abstain / refuse), all gold IDs cited, no invented IDs, right language,
no dose numbers.

In [ ]:
import csv
import random
import re
from collections import defaultdict

CITE = re.compile(r"\[(KB-\d{3}-(?:LM|HI))\]")
SINHALA = re.compile(r"[඀-෿]")
DOSE = re.compile(r"\b\d+(?:\.\d+)?\s*(?:mg|ml|mL|g|cc|iu|IU|units?|mg/kg|ml/kg)\b"
                  r"|withdrawal\s+(?:period|time)\s+(?:of|is)\s+\d", re.I)


def behaviour(text):
    if "සාක්ෂිවල මේ ප්‍රශ්නයට" in text or "does not answer" in text:
        return "abstain"
    if "මාත්‍රා හෝ" in text or "cannot give medicine" in text:
        return "refuse"
    return "answer"


def score(row, out):
    cites = CITE.findall(out)
    allowed = set(CITE.findall(row["prompt"]))
    gold = set(CITE.findall(row["answer"]))
    has_si = bool(SINHALA.search(out))
    return {"behaviour_ok": behaviour(out) == row["kind"],
            "gold_cited": gold <= set(cites) if gold else None,
            "invented_cite": any(c not in allowed for c in cites),
            "lang_ok": has_si if row["lang"] == "si" else not has_si,
            "dose_leak": bool(DOSE.search(out))}


EVAL_N = 80
random.seed(0)
by_group = defaultdict(list)
for r in test_rows:
    by_group[(r["type"], r["lang"])].append(r)
sample = [r for group in by_group.values()
          for r in random.sample(group, min(len(group), max(2, EVAL_N // len(by_group))))]

FastLanguageModel.for_inference(model)
results = []
for i, r in enumerate(sample):
    ids = tokenizer(chat(r["prompt"]), return_tensors="pt", add_special_tokens=False).to("cuda")
    gen = model.generate(**ids, max_new_tokens=160 if r["lang"] == "en" else 512, do_sample=False,
                         eos_token_id=tokenizer.convert_tokens_to_ids("<|eot_id|>"))
    out = tokenizer.decode(gen[0][ids.input_ids.shape[1]:], skip_special_tokens=True).strip()
    results.append({"type": r["type"], "lang": r["lang"], "question": r["question"],
                    "output": out, **score(r, out)})
    print(i, r["type"], r["lang"], results[-1]["behaviour_ok"], results[-1]["gold_cited"])

with open("v2_eval.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(results[0]))
    writer.writeheader()
    writer.writerows(results)
for key in ("behaviour_ok", "gold_cited", "invented_cite", "lang_ok", "dose_leak"):
    for lang in ("en", "si"):
        vals = [x[key] for x in results if x["lang"] == lang and x[key] is not None]
        print(f"{key:14} {lang}: {sum(vals) / len(vals):.2f}  (n={len(vals)})")

## 5. Save to Google Drive
The LoRA adapter (small) and the Q4_K_M GGUF (~4.9 GB) for Ollama. Then on the PC:
1. Rename the old `medfarm-llama-Q4_K_M.gguf` to `medfarm-llama-v1-Q4_K_M.gguf` (keep it to compare).
2. Put the new GGUF in `backend/models/chat_bot/` as `medfarm-llama-Q4_K_M.gguf`.
3. `ollama rm medfarm-llama` then `ollama create medfarm-llama -f Modelfile`.

In [ ]:
import glob
import shutil
from google.colab import drive

drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/medfarm_v2"
model.save_pretrained("hf_lora_adapter_v2")
tokenizer.save_pretrained("hf_lora_adapter_v2")
shutil.copytree("hf_lora_adapter_v2", f"{OUT}/hf_lora_adapter_v2", dirs_exist_ok=True)
shutil.copy("v2_eval.csv", OUT)
json.dump({"train_examples": len(train_rows), "test_examples": len(test_rows),
           "max_len": MAX_LEN, "epochs": 1, "lr": 2e-4, "eff_batch": 8,
           "train_loss": stats.training_loss, "log": trainer.state.log_history},
          open(f"{OUT}/train_log_v2.json", "w"), indent=1)

model.save_pretrained_gguf("medfarm_gguf", tokenizer, quantization_method="q4_k_m")
gguf = [p for p in glob.glob("medfarm_gguf*/*.gguf") + glob.glob("*.gguf") if "Q4_K_M" in p.upper()]
print(gguf)
shutil.copy(gguf[0], f"{OUT}/medfarm-llama-Q4_K_M.gguf")
print("saved to", OUT)